In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

print("Libraries imported successfully.")

Libraries imported successfully.


MySQL Connection

In [17]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import getpass

# MySQL connection settings
username = "root"
password = quote_plus(getpass.getpass("Enter MySQL root password: "))
host = "localhost"
port = 3306

# Source OLTP database
oltp_database = "sakila"

# Target Data Warehouse database
dw_database = "movie_rental_dw"

# Create SQLAlchemy engines
oltp_engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{oltp_database}"
)

dw_engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{dw_database}"
)

print("Connection engines created successfully.")

Connection engines created successfully.


Test Connection

In [18]:
# Test source database connection
test_oltp = pd.read_sql("SELECT VERSION() AS mysql_version;", oltp_engine)

# Test data warehouse connection
test_dw = pd.read_sql("SELECT DATABASE() AS current_database;", dw_engine)

display(test_oltp)
display(test_dw)

,mysql_version
0,8.0.46


,current_database
0,movie_rental_dw


Read OLTP Tables from sakila

In [4]:
# Read main OLTP tables from sakila database

rental_df = pd.read_sql("SELECT * FROM rental", oltp_engine)
payment_df = pd.read_sql("SELECT * FROM payment", oltp_engine)
customer_df = pd.read_sql("SELECT * FROM customer", oltp_engine)
film_df = pd.read_sql("SELECT * FROM film", oltp_engine)
inventory_df = pd.read_sql("SELECT * FROM inventory", oltp_engine)
store_df = pd.read_sql("SELECT * FROM store", oltp_engine)
staff_df = pd.read_sql("SELECT * FROM staff", oltp_engine)

address_df = pd.read_sql("SELECT * FROM address", oltp_engine)
city_df = pd.read_sql("SELECT * FROM city", oltp_engine)
country_df = pd.read_sql("SELECT * FROM country", oltp_engine)

language_df = pd.read_sql("SELECT * FROM language", oltp_engine)
category_df = pd.read_sql("SELECT * FROM category", oltp_engine)
film_category_df = pd.read_sql("SELECT * FROM film_category", oltp_engine)

print("All OLTP tables were loaded successfully into Pandas DataFrames.")

All OLTP tables were loaded successfully into Pandas DataFrames.


Check Row Counts

In [5]:
tables_summary = pd.DataFrame({
    "table_name": [
        "rental", "payment", "customer", "film", "inventory",
        "store", "staff", "address", "city", "country",
        "language", "category", "film_category"
    ],
    "row_count": [
        len(rental_df), len(payment_df), len(customer_df), len(film_df), len(inventory_df),
        len(store_df), len(staff_df), len(address_df), len(city_df), len(country_df),
        len(language_df), len(category_df), len(film_category_df)
    ]
})

tables_summary

,table_name,row_count
0,rental,16044
1,payment,16044
2,customer,599
3,film,1000
4,inventory,4581
5,store,2
6,staff,2
7,address,603
8,city,600
9,country,109


Preview Important Tables

In [6]:
rental_df.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [7]:
payment_df.head()

,payment_id,customer_id,staff_id,rental_id,amount,payment_date,last_update
0,1,1,1,76,2.99,2005-05-25 11:30:37,2006-02-15 22:12:30
1,2,1,1,573,0.99,2005-05-28 10:35:23,2006-02-15 22:12:30
2,3,1,1,1185,5.99,2005-06-15 00:54:12,2006-02-15 22:12:30
3,4,1,2,1422,0.99,2005-06-15 18:02:53,2006-02-15 22:12:30
4,5,1,2,1476,9.99,2005-06-15 21:08:46,2006-02-15 22:12:30


In [8]:
film_df.head()

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2006-02-15 05:03:42
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2006-02-15 05:03:42
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,1,None,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes",2006-02-15 05:03:42
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,1,None,6,2.99,130,22.99,G,Deleted Scenes,2006-02-15 05:03:42


Basic Data Quality Check

In [9]:
quality_summary = pd.DataFrame({
    "table_name": [
        "rental", "payment", "customer", "film", "inventory",
        "store", "staff"
    ],
    "rows": [
        len(rental_df), len(payment_df), len(customer_df), len(film_df),
        len(inventory_df), len(store_df), len(staff_df)
    ],
    "columns": [
        rental_df.shape[1], payment_df.shape[1], customer_df.shape[1],
        film_df.shape[1], inventory_df.shape[1], store_df.shape[1], staff_df.shape[1]
    ],
    "duplicate_rows": [
        rental_df.duplicated().sum(), payment_df.duplicated().sum(),
        customer_df.duplicated().sum(), film_df.duplicated().sum(),
        inventory_df.duplicated().sum(), store_df.duplicated().sum(),
        staff_df.duplicated().sum()
    ],
    "missing_values": [
        rental_df.isnull().sum().sum(), payment_df.isnull().sum().sum(),
        customer_df.isnull().sum().sum(), film_df.isnull().sum().sum(),
        inventory_df.isnull().sum().sum(), store_df.isnull().sum().sum(),
        staff_df.isnull().sum().sum()
    ]
})

quality_summary

,table_name,rows,columns,duplicate_rows,missing_values
0,rental,16044,7,0,183
1,payment,16044,7,0,0
2,customer,599,9,0,0
3,film,1000,13,0,1000
4,inventory,4581,4,0,0
5,store,2,4,0,0
6,staff,2,11,0,2


Build dim_date

In [10]:
# Convert date columns to datetime
rental_df["rental_date"] = pd.to_datetime(rental_df["rental_date"])
rental_df["return_date"] = pd.to_datetime(rental_df["return_date"])
payment_df["payment_date"] = pd.to_datetime(payment_df["payment_date"])

# Collect all dates from rental and payment tables
all_dates = pd.concat([
    rental_df["rental_date"].dt.date,
    rental_df["return_date"].dropna().dt.date,
    payment_df["payment_date"].dt.date
]).dropna().drop_duplicates().sort_values()

# Create dim_date DataFrame
dim_date = pd.DataFrame({
    "full_date": pd.to_datetime(all_dates)
})

# Create date_key in YYYYMMDD format
dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["day_number"] = dim_date["full_date"].dt.day
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["month_number"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["quarter_number"] = dim_date["full_date"].dt.quarter
dim_date["year_number"] = dim_date["full_date"].dt.year

# Reorder columns to match MySQL table
dim_date = dim_date[
    [
        "date_key",
        "full_date",
        "day_number",
        "day_name",
        "month_number",
        "month_name",
        "quarter_number",
        "year_number"
    ]
]

dim_date.head()

,date_key,full_date,day_number,day_name,month_number,month_name,quarter_number,year_number
0,20050524,2005-05-24,24,Tuesday,5,May,2,2005
8,20050525,2005-05-25,25,Wednesday,5,May,2,2005
145,20050526,2005-05-26,26,Thursday,5,May,2,2005
319,20050527,2005-05-27,27,Friday,5,May,2,2005
485,20050528,2005-05-28,28,Saturday,5,May,2,2005


Load dim_date

In [11]:
dim_date.to_sql(
    name="dim_date",
    con=dw_engine,
    if_exists="append",
    index=False
)

print("dim_date loaded successfully.")
print("Rows loaded:", len(dim_date))

dim_date loaded successfully.
Rows loaded: 90


Verify dim_date

In [12]:
pd.read_sql("SELECT COUNT(*) AS dim_date_rows FROM dim_date;", dw_engine)

,dim_date_rows
0,90


Read Customer Location Tables

In [13]:
customer_df = pd.read_sql("SELECT * FROM customer", oltp_engine)
address_df = pd.read_sql("SELECT * FROM address", oltp_engine)
city_df = pd.read_sql("SELECT * FROM city", oltp_engine)
country_df = pd.read_sql("SELECT * FROM country", oltp_engine)

print("Customer-related tables loaded successfully.")
print("Customers:", len(customer_df))
print("Addresses:", len(address_df))
print("Cities:", len(city_df))
print("Countries:", len(country_df))

Customer-related tables loaded successfully.
Customers: 599
Addresses: 603
Cities: 600
Countries: 109


Build dim_customer

In [14]:
dim_customer = (
    customer_df
    .merge(address_df, on="address_id", how="left", suffixes=("", "_address"))
    .merge(city_df, on="city_id", how="left", suffixes=("", "_city"))
    .merge(country_df, on="country_id", how="left", suffixes=("", "_country"))
)

dim_customer["full_name"] = (
    dim_customer["first_name"].fillna("") + " " + dim_customer["last_name"].fillna("")
).str.strip()

dim_customer = dim_customer[
    [
        "customer_id",
        "full_name",
        "email",
        "active",
        "address",
        "district",
        "city",
        "country",
        "last_update"
    ]
].copy()

dim_customer = dim_customer.rename(columns={
    "last_update": "source_last_update"
})

dim_customer.head()

,customer_id,full_name,email,active,address,district,city,country,source_last_update
0,1,MARY SMITH,MARY.SMITH@sakilacustomer.org,1,1913 Hanoi Way,Nagasaki,Sasebo,Japan,2006-02-15 04:57:20
1,2,PATRICIA JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,1,1121 Loja Avenue,California,San Bernardino,United States,2006-02-15 04:57:20
2,3,LINDA WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,1,692 Joliet Street,Attika,Athenai,Greece,2006-02-15 04:57:20
3,4,BARBARA JONES,BARBARA.JONES@sakilacustomer.org,1,1566 Inegl Manor,Mandalay,Myingyan,Myanmar,2006-02-15 04:57:20
4,5,ELIZABETH BROWN,ELIZABETH.BROWN@sakilacustomer.org,1,53 Idfu Parkway,Nantou,Nantou,Taiwan,2006-02-15 04:57:20


Load dim_customer

In [15]:
dim_customer.to_sql(
    name="dim_customer",
    con=dw_engine,
    if_exists="append",
    index=False
)

print("dim_customer loaded successfully.")
print("Rows loaded:", len(dim_customer))

dim_customer loaded successfully.
Rows loaded: 599


Verify dim_customer

In [16]:
pd.read_sql("SELECT COUNT(*) AS dim_customer_rows FROM dim_customer;", dw_engine)

,dim_customer_rows
0,599


Check dim_film empty

In [19]:
pd.read_sql("SELECT COUNT(*) AS dim_film_rows FROM dim_film;", dw_engine)

,dim_film_rows
0,0


Build dim_film

In [20]:
film_df = pd.read_sql("SELECT * FROM film", oltp_engine)
language_df = pd.read_sql("SELECT * FROM language", oltp_engine)
film_category_df = pd.read_sql("SELECT * FROM film_category", oltp_engine)
category_df = pd.read_sql("SELECT * FROM category", oltp_engine)

dim_film = (
    film_df
    .merge(language_df, on="language_id", how="left", suffixes=("", "_language"))
    .merge(film_category_df, on="film_id", how="left")
    .merge(category_df, on="category_id", how="left", suffixes=("", "_category"))
)

dim_film = dim_film[
    [
        "film_id",
        "title",
        "description",
        "release_year",
        "name",
        "name_category",
        "rental_duration",
        "rental_rate",
        "length",
        "replacement_cost",
        "rating",
        "last_update"
    ]
].copy()

dim_film = dim_film.rename(columns={
    "name": "language_name",
    "name_category": "category_name",
    "last_update": "source_last_update"
})

dim_film.head()

,film_id,title,description,release_year,language_name,category_name,rental_duration,rental_rate,length,replacement_cost,rating,source_last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,English,Documentary,6,0.99,86,20.99,PG,2006-02-15 04:46:27
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,English,Horror,3,4.99,48,12.99,G,2006-02-15 04:46:27
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,English,Documentary,7,2.99,50,18.99,NC-17,2006-02-15 04:46:27
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,English,Horror,5,2.99,117,26.99,G,2006-02-15 04:46:27
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,English,Family,6,2.99,130,22.99,G,2006-02-15 04:46:27


Load dim_film

In [21]:
dim_film.to_sql(
    name="dim_film",
    con=dw_engine,
    if_exists="append",
    index=False
)

print("dim_film loaded successfully.")
print("Rows loaded:", len(dim_film))

dim_film loaded successfully.
Rows loaded: 1000


Verify

In [22]:
pd.read_sql("SELECT COUNT(*) AS dim_film_rows FROM dim_film;", dw_engine)

,dim_film_rows
0,1000


Check dim_store

In [23]:
pd.read_sql("SELECT COUNT(*) AS dim_store_rows FROM dim_store;", dw_engine)

,dim_store_rows
0,0


Build dim_store

In [24]:
store_df = pd.read_sql("SELECT * FROM store", oltp_engine)
address_df = pd.read_sql("SELECT * FROM address", oltp_engine)
city_df = pd.read_sql("SELECT * FROM city", oltp_engine)
country_df = pd.read_sql("SELECT * FROM country", oltp_engine)

dim_store = (
    store_df
    .merge(address_df, on="address_id", how="left", suffixes=("", "_address"))
    .merge(city_df, on="city_id", how="left", suffixes=("", "_city"))
    .merge(country_df, on="country_id", how="left", suffixes=("", "_country"))
)

dim_store = dim_store[
    [
        "store_id",
        "manager_staff_id",
        "address",
        "district",
        "city",
        "country",
        "last_update"
    ]
].copy()

dim_store = dim_store.rename(columns={
    "last_update": "source_last_update"
})

dim_store

,store_id,manager_staff_id,address,district,city,country,source_last_update
0,1,1,47 MySakila Drive,Alberta,Lethbridge,Canada,2006-02-15 04:57:12
1,2,2,28 MySQL Boulevard,QLD,Woodridge,Australia,2006-02-15 04:57:12


Load dim_store

In [25]:
dim_store.to_sql(
    name="dim_store",
    con=dw_engine,
    if_exists="append",
    index=False
)

print("dim_store loaded successfully.")
print("Rows loaded:", len(dim_store))

dim_store loaded successfully.
Rows loaded: 2


Verify dim_store

In [26]:
pd.read_sql("SELECT COUNT(*) AS dim_store_rows FROM dim_store;", dw_engine)

,dim_store_rows
0,2


Check dim_staff

In [27]:
pd.read_sql("SELECT COUNT(*) AS dim_staff_rows FROM dim_staff;", dw_engine)

,dim_staff_rows
0,0


Build dim_staff

In [28]:
staff_df = pd.read_sql("SELECT * FROM staff", oltp_engine)
address_df = pd.read_sql("SELECT * FROM address", oltp_engine)
city_df = pd.read_sql("SELECT * FROM city", oltp_engine)
country_df = pd.read_sql("SELECT * FROM country", oltp_engine)

dim_staff = (
    staff_df
    .merge(address_df, on="address_id", how="left", suffixes=("", "_address"))
    .merge(city_df, on="city_id", how="left", suffixes=("", "_city"))
    .merge(country_df, on="country_id", how="left", suffixes=("", "_country"))
)

dim_staff["full_name"] = (
    dim_staff["first_name"].fillna("") + " " + dim_staff["last_name"].fillna("")
).str.strip()

dim_staff = dim_staff[
    [
        "staff_id",
        "full_name",
        "email",
        "active",
        "store_id",
        "address",
        "city",
        "country",
        "last_update"
    ]
].copy()

dim_staff = dim_staff.rename(columns={
    "last_update": "source_last_update"
})

dim_staff

,staff_id,full_name,email,active,store_id,address,city,country,source_last_update
0,1,Mike Hillyer,Mike.Hillyer@sakilastaff.com,1,1,23 Workhaven Lane,Lethbridge,Canada,2006-02-15 03:57:16
1,2,Jon Stephens,Jon.Stephens@sakilastaff.com,1,2,1411 Lillydale Drive,Woodridge,Australia,2006-02-15 03:57:16


Load dim_staff

In [29]:
dim_staff.to_sql(
    name="dim_staff",
    con=dw_engine,
    if_exists="append",
    index=False
)

print("dim_staff loaded successfully.")
print("Rows loaded:", len(dim_staff))

dim_staff loaded successfully.
Rows loaded: 2


Verify dim_staff

In [30]:
pd.read_sql("SELECT COUNT(*) AS dim_staff_rows FROM dim_staff;", dw_engine)

,dim_staff_rows
0,2


## fact_inventory_snapshot...

1. Check fact_inventory_snapshot

In [31]:
pd.read_sql("SELECT COUNT(*) AS fact_inventory_snapshot_rows FROM fact_inventory_snapshot;", dw_engine)

,fact_inventory_snapshot_rows
0,0


2. Build fact_inventory_snapshot

In [32]:
inventory_df = pd.read_sql("SELECT * FROM inventory", oltp_engine)

dim_film_db = pd.read_sql("SELECT film_key, film_id FROM dim_film", dw_engine)
dim_store_db = pd.read_sql("SELECT store_key, store_id FROM dim_store", dw_engine)

fact_inventory_snapshot = (
    inventory_df
    .merge(dim_film_db, on="film_id", how="left")
    .merge(dim_store_db, on="store_id", how="left")
)

fact_inventory_snapshot = (
    fact_inventory_snapshot
    .groupby(["film_key", "store_key"], as_index=False)
    .agg(inventory_count=("inventory_id", "count"))
)

fact_inventory_snapshot.head()

,film_key,store_key,inventory_count
0,1,1,4
1,1,2,4
2,2,2,3
3,3,2,4
4,4,1,4


3. Check nulls before loading

In [33]:
fact_inventory_snapshot.isnull().sum()

film_key           0
store_key          0
inventory_count    0
dtype: int64

4. Load fact_inventory_snapshot

In [34]:
fact_inventory_snapshot.to_sql(
    name="fact_inventory_snapshot",
    con=dw_engine,
    if_exists="append",
    index=False
)

print("fact_inventory_snapshot loaded successfully.")
print("Rows loaded:", len(fact_inventory_snapshot))

fact_inventory_snapshot loaded successfully.
Rows loaded: 1521


5. Verify

In [35]:
pd.read_sql(
    "SELECT COUNT(*) AS fact_inventory_snapshot_rows FROM fact_inventory_snapshot;",
    dw_engine
)

,fact_inventory_snapshot_rows
0,1521


## fact_rental...

1. Check fact_rental empty

In [36]:
pd.read_sql("SELECT COUNT(*) AS fact_rental_rows FROM fact_rental;", dw_engine)

,fact_rental_rows
0,0


2. Build fact_rental

In [37]:
# Read required OLTP tables
rental_df = pd.read_sql("SELECT * FROM rental", oltp_engine)
inventory_df = pd.read_sql("SELECT * FROM inventory", oltp_engine)
film_df = pd.read_sql("SELECT film_id, rental_duration FROM film", oltp_engine)

# Read dimension lookup tables from DW
dim_customer_db = pd.read_sql("SELECT customer_key, customer_id FROM dim_customer", dw_engine)
dim_film_db = pd.read_sql("SELECT film_key, film_id FROM dim_film", dw_engine)
dim_store_db = pd.read_sql("SELECT store_key, store_id FROM dim_store", dw_engine)
dim_staff_db = pd.read_sql("SELECT staff_key, staff_id FROM dim_staff", dw_engine)

# Convert date columns
rental_df["rental_date"] = pd.to_datetime(rental_df["rental_date"])
rental_df["return_date"] = pd.to_datetime(rental_df["return_date"])

# Join rental with inventory to get film_id and store_id
fact_rental = (
    rental_df
    .merge(inventory_df[["inventory_id", "film_id", "store_id"]], on="inventory_id", how="left")
    .merge(film_df, on="film_id", how="left")
    .merge(dim_customer_db, on="customer_id", how="left")
    .merge(dim_film_db, on="film_id", how="left")
    .merge(dim_store_db, on="store_id", how="left")
    .merge(dim_staff_db, on="staff_id", how="left")
)

# Create date keys
fact_rental["rental_date_key"] = fact_rental["rental_date"].dt.strftime("%Y%m%d").astype(int)
fact_rental["return_date_key"] = fact_rental["return_date"].dt.strftime("%Y%m%d")
fact_rental["return_date_key"] = pd.to_numeric(fact_rental["return_date_key"], errors="coerce").astype("Int64")

# Measures
fact_rental["rental_count"] = 1
fact_rental["rental_duration_days"] = (
    fact_rental["return_date"] - fact_rental["rental_date"]
).dt.days

fact_rental["expected_rental_duration"] = fact_rental["rental_duration"]

fact_rental["late_days"] = (
    fact_rental["rental_duration_days"] - fact_rental["expected_rental_duration"]
)

fact_rental["late_days"] = fact_rental["late_days"].apply(
    lambda x: x if pd.notnull(x) and x > 0 else 0
)

fact_rental["late_return_flag"] = fact_rental["late_days"].apply(
    lambda x: True if x > 0 else False
)

# Select final columns matching MySQL fact_rental table
fact_rental = fact_rental[
    [
        "rental_id",
        "rental_date_key",
        "return_date_key",
        "customer_key",
        "film_key",
        "store_key",
        "staff_key",
        "rental_count",
        "rental_duration_days",
        "expected_rental_duration",
        "late_return_flag",
        "late_days"
    ]
].copy()

fact_rental.head()

,rental_id,rental_date_key,return_date_key,customer_key,film_key,store_key,staff_key,rental_count,rental_duration_days,expected_rental_duration,late_return_flag,late_days
0,1,20050524,20050526,130,80,1,1,1,1.0,7,False,0.0
1,2,20050524,20050528,459,333,2,1,1,3.0,7,False,0.0
2,3,20050524,20050601,408,373,2,1,1,7.0,7,False,0.0
3,4,20050524,20050603,333,535,1,2,1,9.0,6,True,3.0
4,5,20050524,20050602,222,450,2,1,1,8.0,5,True,3.0


3. Check nulls before loading

In [38]:
fact_rental.isnull().sum()

rental_id                     0
rental_date_key               0
return_date_key             183
customer_key                  0
film_key                      0
store_key                     0
staff_key                     0
rental_count                  0
rental_duration_days        183
expected_rental_duration      0
late_return_flag              0
late_days                     0
dtype: int64

4. Load fact_rental

In [39]:
fact_rental.to_sql(
    name="fact_rental",
    con=dw_engine,
    if_exists="append",
    index=False
)

print("fact_rental loaded successfully.")
print("Rows loaded:", len(fact_rental))

fact_rental loaded successfully.
Rows loaded: 16044


## fact_payment

1. Check fact_payment empty

In [40]:
pd.read_sql("SELECT COUNT(*) AS fact_payment_rows FROM fact_payment;", dw_engine)

,fact_payment_rows
0,0


2. Build fact_payment

In [41]:
# Read required OLTP tables
payment_df = pd.read_sql("SELECT * FROM payment", oltp_engine)
rental_df = pd.read_sql("SELECT rental_id, inventory_id FROM rental", oltp_engine)
inventory_df = pd.read_sql("SELECT inventory_id, film_id, store_id FROM inventory", oltp_engine)

# Read dimension lookup tables from DW
dim_customer_db = pd.read_sql("SELECT customer_key, customer_id FROM dim_customer", dw_engine)
dim_staff_db = pd.read_sql("SELECT staff_key, staff_id FROM dim_staff", dw_engine)
dim_film_db = pd.read_sql("SELECT film_key, film_id FROM dim_film", dw_engine)
dim_store_db = pd.read_sql("SELECT store_key, store_id FROM dim_store", dw_engine)

# Convert payment date to datetime
payment_df["payment_date"] = pd.to_datetime(payment_df["payment_date"])

# Join payment with rental and inventory to get film and store context
fact_payment = (
    payment_df
    .merge(rental_df, on="rental_id", how="left")
    .merge(inventory_df, on="inventory_id", how="left")
    .merge(dim_customer_db, on="customer_id", how="left")
    .merge(dim_staff_db, on="staff_id", how="left")
    .merge(dim_film_db, on="film_id", how="left")
    .merge(dim_store_db, on="store_id", how="left")
)

# Create date key
fact_payment["payment_date_key"] = fact_payment["payment_date"].dt.strftime("%Y%m%d").astype(int)

# Measures
fact_payment["payment_count"] = 1
fact_payment["payment_amount"] = fact_payment["amount"]

# Select final columns matching MySQL fact_payment table
fact_payment = fact_payment[
    [
        "payment_id",
        "payment_date_key",
        "customer_key",
        "staff_key",
        "film_key",
        "store_key",
        "rental_id",
        "payment_count",
        "payment_amount"
    ]
].copy()

fact_payment.head()

,payment_id,payment_date_key,customer_key,staff_key,film_key,store_key,rental_id,payment_count,payment_amount
0,1,20050525,1,1,663,2,76,1,2.99
1,2,20050528,1,1,875,2,573,1,0.99
2,3,20050615,1,1,611,1,1185,1,5.99
3,4,20050615,1,2,228,2,1422,1,0.99
4,5,20050615,1,2,308,1,1476,1,9.99


3. Check nulls

In [42]:
fact_payment.isnull().sum()

payment_id          0
payment_date_key    0
customer_key        0
staff_key           0
film_key            0
store_key           0
rental_id           0
payment_count       0
payment_amount      0
dtype: int64

4. Load fact_payment

In [43]:
fact_payment.to_sql(
    name="fact_payment",
    con=dw_engine,
    if_exists="append",
    index=False
)

print("fact_payment loaded successfully.")
print("Rows loaded:", len(fact_payment))

fact_payment loaded successfully.
Rows loaded: 16044


5. Verify

In [44]:
pd.read_sql(
    "SELECT COUNT(*) AS fact_payment_rows FROM fact_payment;",
    dw_engine
)

,fact_payment_rows
0,16044


## Final Verification...

In [45]:
final_dw_summary = pd.read_sql("""
SELECT 'dim_date' AS table_name, COUNT(*) AS row_count FROM dim_date
UNION ALL
SELECT 'dim_customer', COUNT(*) FROM dim_customer
UNION ALL
SELECT 'dim_film', COUNT(*) FROM dim_film
UNION ALL
SELECT 'dim_store', COUNT(*) FROM dim_store
UNION ALL
SELECT 'dim_staff', COUNT(*) FROM dim_staff
UNION ALL
SELECT 'fact_inventory_snapshot', COUNT(*) FROM fact_inventory_snapshot
UNION ALL
SELECT 'fact_rental', COUNT(*) FROM fact_rental
UNION ALL
SELECT 'fact_payment', COUNT(*) FROM fact_payment;
""", dw_engine)

final_dw_summary

,table_name,row_count
0,dim_date,90
1,dim_customer,599
2,dim_film,1000
3,dim_store,2
4,dim_staff,2
5,fact_inventory_snapshot,1521
6,fact_rental,16044
7,fact_payment,16044


In [46]:
fact_quality_check = pd.read_sql("""
SELECT 
    'fact_rental' AS fact_table,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS missing_customer_key,
    SUM(CASE WHEN film_key IS NULL THEN 1 ELSE 0 END) AS missing_film_key,
    SUM(CASE WHEN store_key IS NULL THEN 1 ELSE 0 END) AS missing_store_key,
    SUM(CASE WHEN staff_key IS NULL THEN 1 ELSE 0 END) AS missing_staff_key
FROM fact_rental

UNION ALL

SELECT 
    'fact_payment' AS fact_table,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS missing_customer_key,
    SUM(CASE WHEN film_key IS NULL THEN 1 ELSE 0 END) AS missing_film_key,
    SUM(CASE WHEN store_key IS NULL THEN 1 ELSE 0 END) AS missing_store_key,
    SUM(CASE WHEN staff_key IS NULL THEN 1 ELSE 0 END) AS missing_staff_key
FROM fact_payment;
""", dw_engine)

fact_quality_check

,fact_table,total_rows,missing_customer_key,missing_film_key,missing_store_key,missing_staff_key
0,fact_rental,16044,0.0,0.0,0.0,0.0
1,fact_payment,16044,0.0,0.0,0.0,0.0


## Analytical Business Questions

Q1: Which films are rented most frequently?

In [47]:
top_rented_films = pd.read_sql("""
SELECT 
    f.title,
    f.category_name,
    SUM(r.rental_count) AS total_rentals
FROM fact_rental r
JOIN dim_film f ON r.film_key = f.film_key
GROUP BY f.title, f.category_name
ORDER BY total_rentals DESC
LIMIT 10;
""", dw_engine)

top_rented_films

,title,category_name,total_rentals
0,BUCKET BROTHERHOOD,Travel,34.0
1,ROCKETEER MOTHER,Foreign,33.0
2,RIDGEMONT SUBMARINE,New,32.0
3,GRIT CLOCKWORK,Games,32.0
4,SCALAWAG DUCK,Music,32.0
5,JUGGLER HARDLY,Animation,32.0
6,FORWARD TEMPLE,Games,32.0
7,HOBBIT ALIEN,Drama,31.0
8,ROBBERS JOON,Children,31.0
9,ZORRO ARK,Comedy,31.0


Q2: Which films generate the highest revenue?

In [48]:
top_revenue_films = pd.read_sql("""
SELECT 
    f.title,
    f.category_name,
    SUM(p.payment_amount) AS total_revenue
FROM fact_payment p
JOIN dim_film f ON p.film_key = f.film_key
GROUP BY f.title, f.category_name
ORDER BY total_revenue DESC
LIMIT 10;
""", dw_engine)

top_revenue_films

,title,category_name,total_revenue
0,TELEGRAPH VOYAGE,Music,231.73
1,WIFE TURN,Documentary,223.69
2,ZORRO ARK,Comedy,214.69
3,GOODFELLAS SALUTE,Sci-Fi,209.69
4,SATURDAY LAMBS,Sports,204.72
5,TITANS JERK,Sci-Fi,201.71
6,TORQUE BOUND,Drama,198.72
7,HARRY IDAHO,Drama,195.70
8,INNOCENT USUAL,Foreign,191.74
9,HUSTLER PARTY,Comedy,190.78


Q3: How does revenue change by month?

In [49]:
revenue_by_month = pd.read_sql("""
SELECT 
    d.year_number,
    d.month_number,
    d.month_name,
    SUM(p.payment_amount) AS total_revenue
FROM fact_payment p
JOIN dim_date d ON p.payment_date_key = d.date_key
GROUP BY d.year_number, d.month_number, d.month_name
ORDER BY d.year_number, d.month_number;
""", dw_engine)

revenue_by_month

,year_number,month_number,month_name,total_revenue
0,2005,5,May,4823.44
1,2005,6,June,9629.89
2,2005,7,July,28368.91
3,2005,8,August,24070.14
4,2006,2,February,514.18


Q4: Which stores have the highest number of rentals?

In [50]:
rentals_by_store = pd.read_sql("""
SELECT 
    s.store_id,
    s.city,
    s.country,
    SUM(r.rental_count) AS total_rentals
FROM fact_rental r
JOIN dim_store s ON r.store_key = s.store_key
GROUP BY s.store_id, s.city, s.country
ORDER BY total_rentals DESC;
""", dw_engine)

rentals_by_store

,store_id,city,country,total_rentals
0,2,Woodridge,Australia,8121.0
1,1,Lethbridge,Canada,7923.0


Q5: Which customers generate the highest revenue?

In [51]:
top_customers_revenue = pd.read_sql("""
SELECT 
    c.customer_id,
    c.full_name,
    c.city,
    c.country,
    SUM(p.payment_amount) AS total_revenue
FROM fact_payment p
JOIN dim_customer c ON p.customer_key = c.customer_key
GROUP BY c.customer_id, c.full_name, c.city, c.country
ORDER BY total_revenue DESC
LIMIT 10;
""", dw_engine)

top_customers_revenue

,customer_id,full_name,city,country,total_revenue
0,526,KARL SEAL,Cape Coral,United States,221.55
1,148,ELEANOR HUNT,Saint-Denis,Réunion,216.54
2,144,CLARA SHAW,Molodetšno,Belarus,195.58
3,137,RHONDA KENNEDY,Apeldoorn,Netherlands,194.61
4,178,MARION SNYDER,Santa Brbara dOeste,Brazil,194.61
5,459,TOMMY COLLAZO,Qomsheh,Iran,186.62
6,469,WESLEY BULL,Ourense (Orense),Spain,177.60
7,468,TIM CARY,Bijapur,India,175.61
8,236,MARCIA DEAN,Tanza,Philippines,175.58
9,181,ANA BRADLEY,Memphis,United States,174.66


Q6: Which film categories have the most late returns?

In [52]:
late_returns_by_category = pd.read_sql("""
SELECT 
    f.category_name,
    COUNT(*) AS total_rentals,
    SUM(CASE WHEN r.late_return_flag = 1 THEN 1 ELSE 0 END) AS late_returns,
    AVG(r.late_days) AS average_late_days
FROM fact_rental r
JOIN dim_film f ON r.film_key = f.film_key
GROUP BY f.category_name
ORDER BY late_returns DESC;
""", dw_engine)

late_returns_by_category

,category_name,total_rentals,late_returns,average_late_days
0,Sports,1179,529.0,1.1959
1,Sci-Fi,1101,484.0,1.1081
2,Animation,1166,452.0,0.9949
3,Documentary,1050,449.0,1.1419
4,Action,1112,449.0,1.0252
5,New,940,424.0,1.3011
6,Family,1096,411.0,0.9425
7,Drama,1060,403.0,0.9217
8,Comedy,941,396.0,1.1403
9,Games,969,396.0,1.0454
